# Phase 3 Kaggle qualification

Private development-only worker. Start with CPU, then use a separate CUDA run and environment for comparison. All five federated clients stay inside one job. Each contributor operates their own account.

No final-test bank, final seeds, AMP, TF32, DDP or automatic account operation. Actual Kaggle GPU qualification and full-data confirmations remain necessary.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, os, subprocess, sys, time

# Attach the private dataset containing qualification.zip, then set its path.
BUNDLE = Path('/kaggle/input/CHANGE-TO-YOUR-PRIVATE-DATASET/qualification.zip')
EXPECTED_SHA256 = 'f355f1a217d1d82844893df7fc5c1284619ac6e7127c21f9e4d2949990dca34f'
WORK = Path('/kaggle/working/phase3-v3')
DEVICE = 'cpu'  # Change to 'cuda:0' for the separate CUDA qualification candidate.
RUN_FULL_DATA = False  # Enable after synthetic checks; development seed 101 only.

def sha(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''):
            h.update(block)
    return h.hexdigest()

assert BUNDLE.is_file(), 'Attach the private bundle dataset and set BUNDLE'
assert sha(BUNDLE) == EXPECTED_SHA256, 'Wrong bundle bytes'
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    manifest = json.loads(archive.read('SNAPSHOT.json'))
    assert not manifest['final_dispatchable'] and not manifest['final_test_accessed']
    assert set(archive.namelist()) == set(manifest['files']) | {'SNAPSHOT.json'}
    assert len(archive.namelist()) == len(set(archive.namelist()))
    for name, expected in manifest['files'].items():
        p = PurePosixPath(name)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in name and ':' not in name
        target = WORK/name
        assert target.resolve().is_relative_to(WORK.resolve())
        if target.exists():
            assert sha(target) == expected, 'Existing source/input changed: '+name
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(name) as source, target.open('xb') as dest:
                import shutil
                shutil.copyfileobj(source, dest, 8*1024**2)
            assert sha(target) == expected
PROJECT = WORK/'project'
os.chdir(PROJECT)
print('Verified frozen source, worker source and train/development input bytes.')


## Install and verify

The notebook image is not the experiment environment. The setup cell creates Python 3.12.14, NumPy 2.3.5 and Torch 2.7.0 in a separate environment. CUDA uses the cu126 build; CPU uses the CPU build.


In [ ]:
# Enable Internet for setup. The candidate uses its own pinned environment.
ENVIRONMENT = '/kaggle/working/phase3-env-v3-'+DEVICE.replace(':', '-')
setup_started = time.perf_counter()
subprocess.run([sys.executable, '-m', 'kaggle_worker.bootstrap',
                '--device', 'cuda' if DEVICE.startswith('cuda') else 'cpu',
                '--output', ENVIRONMENT], check=True)
PYTHON = ENVIRONMENT+'/bin/python'
ENV = {**os.environ, 'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'OMP_NUM_THREADS': '1',
       'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'PYTHONPATH': str(PROJECT)}
OUTPUT = Path('/kaggle/working/phase3-results-v3')/DEVICE.replace(':', '-')
OUTPUT.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copyfile(Path(ENVIRONMENT)/'.phase3-environment.json', OUTPUT/'environment-lock.json')
setup_receipt = OUTPUT/'setup-cost.json'
if not setup_receipt.exists():
    setup_receipt.write_text(json.dumps({'environment_setup_seconds': time.perf_counter()-setup_started,
                                        'device_candidate': DEVICE}, indent=2))

def run(module, *args):
    index = 1
    while (OUTPUT/f'command-{index:04d}.json').exists() or (OUTPUT/f'command-{index:04d}.log').exists():
        index += 1
    log = OUTPUT/f'command-{index:04d}.log'
    command = [PYTHON, '-m', module, *map(str, args)]
    print('Running', module, 'log:', log, flush=True)
    began = time.perf_counter()
    with log.open('x') as stream:
        result = subprocess.run(command, cwd=PROJECT, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
    receipt = {'module': module, 'argv': command, 'returncode': result.returncode,
               'subprocess_wall_seconds': time.perf_counter()-began,
               'scope': 'Includes cold interpreter imports, input verification, execution and saving; excludes setup',
               'log_sha256': sha(log)}
    (OUTPUT/f'command-{index:04d}.json').write_text(json.dumps(receipt, indent=2))
    print(receipt)
    print('\n'.join(log.read_text().splitlines()[-8:]))
    result.check_returncode()

acceptance = OUTPUT/'synthetic-acceptance.json'
run('kaggle_worker.checks', '--device', DEVICE, '--output', acceptance, '--verify-existing')


## Representative complete worker costs

These pilot defaults benchmark execution, not final selected settings. Record the complete receipt times and evaluate all three learners. Afterwards confirm the selected clean settings on seeds 101–103 on the eventual runtime.


In [ ]:
# Fresh jobs or compatible committed continuations. No final seed CLI exists.
# The receipt covers train validation, stage/round work, importance and saving.
# Evaluation uses the same pinned CPU device for every condition.
if RUN_FULL_DATA:
    for learner in ('FT', 'EI', 'DR'):
        job = OUTPUT/learner
        run('kaggle_worker.worker', '--device', DEVICE, '--learner', learner,
            '--train-package', PROJECT/'inputs/train',
            '--train-manifest-sha256', manifest['train_manifest_sha256'],
            '--output', job, '--budget-seconds', 7200,
            '--reserve-seconds', 180, '--transition-bound-seconds', 900)
        receipts = sorted(job.glob('invocation-*.json'))
        if not receipts or json.loads(receipts[-1].read_text())['status'] != 'complete':
            print(learner, 'stopped at a committed boundary; save outputs and resume this job.')
            continue
        metrics = OUTPUT/(learner+'-development')
        if not metrics.exists():
            run('kaggle_worker.evaluate', '--job', job,
                '--development-package', PROJECT/'inputs/development', '--output', metrics)
else:
    print('Full-data confirmations are disabled. Enable RUN_FULL_DATA after inspecting fixture evidence.')


In [ ]:
# Save Version / Run All, wait for success, and download these saved outputs.
# /kaggle/working is the output location; a checkpoint only in a live session is
# not a durable handoff. Keep failed logs and immutable segments in the export.
import shutil
index = 1
export = Path(f'/kaggle/working/phase3-qualification-output-{index:04d}.zip')
while export.exists():
    index += 1
    export = Path(f'/kaggle/working/phase3-qualification-output-{index:04d}.zip')
shutil.make_archive(str(export.with_suffix('')), 'zip', OUTPUT)
print('Output archive:', export, 'SHA256:', sha(export))
print('These results qualify development execution only; final-study dispatch is still sealed.')
